In [1]:
# D1
import pandas as pd
import numpy as np
from datetime import datetime

from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import numbers

from openpyxl.styles import Alignment
import fitz  # pip install PyMuPDF
import csv
import re

In [2]:
# Abrir el archivo PDF
pdf_path = r"Remittance_D1.pdf"
doc = fitz.open(pdf_path)


In [3]:
# Expresión regular para extraer líneas de facturas
factura_pattern = re.compile(r"RE\s+\d+\s+PMP\d+\s+\d{1,3}(?:\.\d{3})*\s+\d+\s+\d{1,3}(?:\.\d{3})*\s+\d+\s+\d{1,3}(?:\.\d{3})*")


In [4]:
# Lista para almacenar las facturas
facturas = []

In [5]:
# Iterar por cada página y extraer las líneas que coinciden con el patrón
for page in doc:
    text = page.get_text()
    matches = factura_pattern.findall(text)
    for match in matches:
        parts = match.split()
        if len(parts) == 8:
            facturas.append({
                "TD": parts[0],
                "Doc.Interno": parts[1],
                "Nro. de Factura": parts[2],
                "Valor Bruto": parts[3].replace('.', ''),
                "Retenciones": parts[4],
                "IVA": parts[5].replace('.', ''),
                "Desc/Rec": parts[6],
                "Neto Pagado": parts[7].replace('.', '')
            })

In [6]:
# Cerrar el documento PDF
doc.close()

In [7]:
# Exportar a archivo CSV
csv_file = r"C:\Users\Jose-Ricardo.Morales\OneDrive - Unilever\Cartera\PROYECTOS\Cash App Cross Andina\Python\D1.csv"
with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=["TD", "Doc.Interno", "Nro. de Factura", "Valor Bruto", "Retenciones", "IVA", "Desc/Rec", "Neto Pagado"])
    writer.writeheader()
    for factura in facturas:
        writer.writerow(factura)

In [8]:
# Renombrar Columna de facturas del remittance
csv_file = pd.read_csv(r"C:\Users\Jose-Ricardo.Morales\OneDrive - Unilever\Cartera\PROYECTOS\Cash App Cross Andina\Python\D1.csv") 
csv_file = csv_file.rename(columns={
    "Nro. de Factura": "Referencia / Factura",
})

# print(f"Se ha generado el archivo CSV: {csv_file}")

In [9]:
# Lee el file CSV
remittance = pd.read_csv(r"C:\Users\Jose-Ricardo.Morales\OneDrive - Unilever\Cartera\PROYECTOS\Cash App Cross Andina\Python\D1.csv",
    header=[0]    # Encabezados
)

In [10]:
remittance["Referencia / Factura"] = remittance["Nro. de Factura"]

In [11]:
remittance["Importe de factura"] = remittance["Neto Pagado"]

In [12]:
remittance["Importe de factura"] = remittance["Neto Pagado"]

In [13]:
remittance["Tipo de Documento"] = "Factura"

In [14]:
remittance["Importe de factura"] = remittance["Importe de factura"].astype(float)

In [15]:
# Cargar Cartera (FBL5N)
FBL5N = pd.read_excel(
    r"FBL5N_d1.xlsx",
    sheet_name="Sheet1",
    usecols=["Document Type", "Reference", "Amount in local currency", "Reason code", "Document Number", "Text"]
)
# Agregar a la importacion de cartera FBL5N que traiga info cuando: Type = "YE" y RCd = "NRO"
# Filtrar directamente Type == RV y NRO
FBL5N = FBL5N[(FBL5N["Document Type"] == "RV") | (FBL5N["Reason code"] == "NRO")]

# Filtar Columnas Cartera (FBL5N)
# Renombrar columnas
FBL5N = FBL5N.rename(columns={
    "Reference": "Referencia / Factura",
    "Amount in local currency": "importe_FBL5N"
}).reset_index(drop=True)
FBL5N["Referencia / Factura"] = np.where(
    FBL5N["Reason code"] == "NRO",
    FBL5N["Document Number"].astype("Int64").astype(str),  # quita el .0
    FBL5N["Referencia / Factura"]
)

In [16]:
# Cruce entre Remittance y FBL5N por "Referencia / Factura"
hrc_template = pd.merge(
    remittance,
    FBL5N,
    on="Referencia / Factura",
    how="left"   # mantiene todas las filas de remittance
)

In [17]:
# Inicializamos la columna como NaN
hrc_template["Diferencia"] = pd.NA

# Calculamos la diferencia solo para facturas
hrc_template.loc[hrc_template["Tipo de Documento"] == "Factura", "Diferencia"] = (
     hrc_template["importe_FBL5N"] - hrc_template["Importe de factura"]
)

/var/folders/zh/b4hc884n7v95b292z7f5fx0w0000gn/T/ipykernel_14852/3749057274.py:5: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  hrc_template.loc[hrc_template["Tipo de Documento"] == "Factura", "Diferencia"] = (


In [18]:
# Filtramos filas donde Diferencia no es NA y distinta de 0 (esto cambia con respecto al Template, menos dos registros)
diferencias = hrc_template[hrc_template["Diferencia"].notna() & (hrc_template["Diferencia"] != 0)].copy()

# Creamos las nuevas filas según tus reglas
registros_diferencias_entre_remittence_cartera = pd.DataFrame({
    "Tipo de Documento": "Descuentos no asociados a FC",
    "Referencia / Factura": diferencias["Referencia / Factura"],
    "Importe de factura": diferencias["Diferencia"],
    "Pago Neto": "",  # opcional
    "Descuento": "MENORES VALORES",
    "Motivo del descuento": diferencias["Diferencia"].apply(
        lambda x: "WOB" if -20000 < x < 0 
        else ("384" if 0 < x < 20000 else "987")
    )
})


In [19]:
# Concatenamos las nuevas filas al DataFrame original
hrc_template = pd.concat([hrc_template, registros_diferencias_entre_remittence_cartera], ignore_index=True)

In [20]:
hrc_template["Comentarios"] = np.where(
    hrc_template["Tipo de Documento"] == "Factura", "",
    np.where(
        hrc_template["Descuento"] == "MENORES VALORES",
        hrc_template["Descuento"],
        hrc_template["Descuento"].fillna("") + " " + hrc_template["Referencia / Factura"].fillna("")
    )
)

# Reglas especiales (pisan el resultado anterior solo cuando aplican)
cond_1 = (hrc_template["Motivo del descuento"] == "987") & (hrc_template["Importe de factura"] < -20000)
cond_2 = (hrc_template["Motivo del descuento"] == "987") & (hrc_template["Importe de factura"] > 20000)

hrc_template.loc[cond_1, "Comentarios"] = "Myr Vlr Pagado " + hrc_template.loc[cond_1, "Referencia / Factura"].fillna("")
hrc_template.loc[cond_2, "Comentarios"] = "Saldo FC " + hrc_template.loc[cond_2, "Referencia / Factura"].fillna("")


In [21]:
# Agrego dato Pago Neto
hrc_template["Pago Neto"] = hrc_template["Importe de factura"]

# Agrego dato NRO (Nota de credito)
# Agrego registros FBL5N con Reason code = NRO que no cruzaron
nota_credito = FBL5N[FBL5N["Reason code"] == "NRO"].copy()

# Forzamos el valor de "Tipo de Documento"
nota_credito["Tipo de Documento"] = "Nota de Crédito"

# Concatenamos
hrc_template = pd.concat([hrc_template, nota_credito], ignore_index=True)

In [22]:
# Limpio las columnas con las que me voy a quedar en Template ordenadas

columnas_finales = [
    "Tipo de Documento",
    "Referencia / Factura",
    "Importe de factura",
    "Descuento",
    "Motivo del descuento",
    "Pago Neto",
    "Comentarios"
]

hrc_template = hrc_template[columnas_finales]

ruta_salida = "Template_HRC_D1.xlsx"

# Exportamos con pandas, indicando hoja y posición inicial
hrc_template.to_excel(
    ruta_salida,
    index=False,
    sheet_name="Template",
    startrow=17,
    startcol=2
)
# Abrimos el archivo para aplicar formatos y cuadros
wb = load_workbook(ruta_salida)
ws = wb["Template"]

# Titulos Template
ws["C2"] = "Desglose de Pago"
ws["C4"] = "CAMPOS NO EDITABLES"
# Cuadro REFERENCIA DE PAGO
ws["G2"] = "REFERENCIA DE PAGO"
# ws["H2"] =  numero_orden # --- Dato dinámico ---
# Cuadro Informacion clinete
ws["C6"] = "Cliente" 
ws["C8"] = "Codigo de Cliente" 
# ws["D6"] = # nombre_cliente # --- Dato dinámico ---
# ws["D8"] = #id_cliente # --- Dato dinámico ---
# Cuadro Datos del pago
ws["C11"] = "Referencia"
# ws["C12"] = numero_orden # --- Dato dinámico ---
ws["D11"] = "Fecha"
# ws["D12"] = #fecha_pago # --- Dato dinámico ---
ws["E11"] = "Método de Pago"
ws["E12"] = "Transferencia"
ws["F11"] = "Valor"
# ws["F12"] = #importe_FBL3N # --- Dato dinámico ---
# Cuadro de montos
ws["F6"] = "TOTAL s/ BANCOS"
# ws["G6"] = #importe_FBL3N # --- Dato dinámico ---
ws["F7"] = "TOTAL s/ DETALLE"
# ws["G7"] = #total_pago_neto # --- Dato dinámico ---
ws["F8"] = "DIFERENCIA"
# ws["G8"] = #-diferencia # --- Dato dinámico ---
# Formato del Template:

# Datos numericos en cuadros
for cell in ["G6", "G7", "G8", "F12"]:
    ws[cell].number_format = '#,##0.00'

# Formato Tabla principal
# Columnas numéricas
num_cols = ["Importe de factura", "Pago Neto"]

for col in num_cols:
    col_idx = hrc_template.columns.get_loc(col) + 3  # startcol=2 → columna C = 3 en openpyxl
    for row in range(18, 18 + len(hrc_template) + 1):  # largo de df +1
        ws.cell(row=row, column=col_idx).number_format = '#,##0.00'
        
# Columnas de texto
## Columnas de texto formateadas y centradas (excepto 'Comentarios')
str_cols = ["Tipo de Documento", "Referencia / Factura", "Descuento", "Motivo del descuento", "Comentarios"]

for col in str_cols:
    col_idx = hrc_template.columns.get_loc(col) + 3
    for row in range(18, 18 + len(hrc_template) + 1):  # largo de df +1
        cell = ws.cell(row=row, column=col_idx)
        cell.number_format = '@'  # formato texto
        if col != "Comentarios":
            cell.alignment = Alignment(horizontal="center", vertical="center")
# Guardar cambios en el archivo Excel
wb.save(ruta_salida)
print(f"Archivo exportado correctamente con formato: {ruta_salida}")
#Archivo exportado correctamente con formato: Template_HRC_Farmatodo.xlsx

Archivo exportado correctamente con formato: Template_HRC_D1.xlsx
